<a href="https://colab.research.google.com/github/derektorquette/reconhecimento-de-emocoes-com-tensorflow-e-python/blob/main/deteccao_de_emocoes_em_videos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Detecção de Emoções em Vídeos

In [ ]:
import cv2
import numpy as np
import pandas as pd
import time
from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt
import zipfile
cv2.__version__

'4.13.0'

In [ ]:
!pip install tf-keras

In [ ]:
import tensorflow as tf
print(tf.__version__)


2.20.0


In [ ]:
from google.colab import drive # Montando o Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

from tf_keras.models import load_model

path = "/content/drive/MyDrive/CURSOS LIVRES FORMAÇÃO COMPLEMENTAR/Reconhecimento de Emoções com TensorFlow 2.0 e Python/Material/Material"

model = load_model(path + "/modelo_02_expressoes.h5", compile=False)
print("Modelo carregado com sucesso!")

Modelo carregado com sucesso!


In [ ]:
arquivo_video = "/content/drive/MyDrive/CURSOS LIVRES FORMAÇÃO COMPLEMENTAR/Reconhecimento de Emoções com TensorFlow 2.0 e Python/Material/Material/Videos/video_teste04.mp4"
cap = cv2.VideoCapture(arquivo_video)

conectado, video = cap.read()
print(conectado, video.shape)

True (360, 640, 3)


In [ ]:
redimensionar = True
largura_maxima = 600

if (redimensionar and video.shape[1] > largura_maxima):
  proporcao = video.shape [1] / video.shape[0]
  video_largura = largura_maxima
  video_altura = int(video_largura / proporcao)
else:
  video_largura = video.shape[1]
  video_altura = video.shape[0]


In [ ]:
nome_arquivo = 'resultado_video_teste04.avi'

fourcc = cv2.VideoWriter_fourcc(*'XVID')

fps = 24

saida_video = cv2.VideoWriter(nome_arquivo, fourcc, fps, (video_largura, video_altura))

In [ ]:
from tensorflow.keras.preprocessing.image import img_to_array

haarcascade_faces = '/content/drive/MyDrive/CURSOS LIVRES FORMAÇÃO COMPLEMENTAR/Reconhecimento de Emoções com TensorFlow 2.0 e Python/Material/Material/haarcascade_frontalface_default.xml'
fonte_pequena, fonte_media = 0.4, 0.7
fonte = cv2.FONT_HERSHEY_SIMPLEX
expressoes = ["Raiva", "Nojo", "Medo", "Feliz", "Triste", "Surpreso", "Neutro"]

while (cv2.waitKey(1) < 0):
  conectado, frame = cap.read()

  if not conectado:
    break

  # Calcular quanto tempo levou para executar as operações
  t = time.time()

  if redimensionar:
    frame = cv2.resize(frame, (video_largura, video_altura))

  face_cascade = cv2.CascadeClassifier(haarcascade_faces)
  cinza = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
  faces = face_cascade.detectMultiScale(cinza, scaleFactor=1.2, minNeighbors=5, minSize=(30, 30))

  if len(faces) > 0:
    for (x, y, w, h) in faces:
      frame = cv2.rectangle(frame, (x, y), (x + w, y + h + 10), (255, 50, 50), 2) # desenho do retângulo na face
      roi = cinza[y:y + h, x:x + w] # região de interesse
      roi = cv2.resize(roi, (48, 48))
      roi = roi.astype("float") / 255.0
      roi = img_to_array(roi)
      roi = np.expand_dims(roi, axis=0)

      # Faz a predição - calcula as probabilidades
      result = model.predict(roi)[0]
      print(result)

      if result is not None:
          resultado = np.argmax(result) # encontra a emoção com maior probabilidade
          cv2.putText(frame, expressoes[resultado], (x, y - 10), fonte, fonte_media, (255, 255, 255), 1, cv2.LINE_AA) # escreve a emoção acima


  cv2.putText(frame, 'Tempo processamento: {:.2f}ms'.format(time.time() - t), (20, video_altura-20), fonte, fonte_pequena, (250, 255, 255), 1, cv2.LINE_AA)

  cv2_imshow(frame)
  saida_video.write(frame) # grava o frame atual

print("Terminou")
saida_video.release()
cv2.destroyAllWindows()


